In [ ]:
#importing packages
from pyspark.sql.functions import *
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [ ]:
#creating sparksessoins
spark = SparkSession.\
builder.\
appName("spark_SCD2").\
getOrCreate()

In [ ]:
target_data = [
    # customer 1 (history)
    (1, "Ram", "Delhi", "2023-01-01", "2024-01-01", "N"),
    (1, "Ram", "Mumbai", "2024-01-01", None, "Y"),

    # customer 2 (no change yet)
    (2, "Shyam", "Mumbai", "2024-01-01", None, "Y"),

    # customer 3 (history)
    (3, "Mohan", "Pune", "2023-06-01", "2024-06-01", "N"),
    (3, "Mohan", "Delhi", "2024-06-01", None, "Y"),

    # customer 4 (only one active)
    (4, "Sita", "Chennai", "2024-03-01", None, "Y"),

    # customer 5 (history)
    (5, "Geeta", "Kolkata", "2023-05-01", "2024-02-01", "N"),
    (5, "Geeta", "Hyderabad", "2024-02-01", None, "Y"),

    # customer 6 (only one active)
    (6, "Ravi", "Bangalore", "2024-07-01", None, "Y")
]

target_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("effective_start_date", StringType(), True),
    StructField("effective_end_date", StringType(), True),
    StructField("active", StringType(), True)
])

target_df = spark.createDataFrame(target_data, target_schema)

target_df.show(truncate=False)

+-----------+-------------+---------+--------------------+------------------+------+
|customer_id|customer_name|city     |effective_start_date|effective_end_date|active|
+-----------+-------------+---------+--------------------+------------------+------+
|1          |Ram          |Delhi    |2023-01-01          |2024-01-01        |N     |
|1          |Ram          |Mumbai   |2024-01-01          |NULL              |Y     |
|2          |Shyam        |Mumbai   |2024-01-01          |NULL              |Y     |
|3          |Mohan        |Pune     |2023-06-01          |2024-06-01        |N     |
|3          |Mohan        |Delhi    |2024-06-01          |NULL              |Y     |
|4          |Sita         |Chennai  |2024-03-01          |NULL              |Y     |
|5          |Geeta        |Kolkata  |2023-05-01          |2024-02-01        |N     |
|5          |Geeta        |Hyderabad|2024-02-01          |NULL              |Y     |
|6          |Ravi         |Bangalore|2024-07-01          |NULL   

In [ ]:
source_data = [
    (1, "Ram", "Bangalore", "2026-04-08"),   # 🔄 changed (Mumbai → Bangalore)
    (2, "Shyam", "Mumbai", "2026-04-08"),    # ✅ same
    (3, "Mohan", "Delhi", "2026-04-08"),     # ✅ same
    (4, "Sita", "Pune", "2026-04-08"),       # 🔄 changed (Chennai → Pune)
    (5, "Geeta", "Hyderabad", "2026-04-08"), # ✅ same
    (6, "Ravi", "Delhi", "2026-04-08"),      # 🔄 changed (Bangalore → Delhi)
    (7, "Amit", "Noida", "2026-04-08"),      # 🆕 new
    (8, "Neha", "Gurgaon", "2026-04-08")     # 🆕 new
]

source_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("food_delivery_address", StringType(), True),
    StructField("sales_date", StringType(), True)
])

source_df = spark.createDataFrame(source_data, source_schema)

source_df.show(truncate=False)

+-----------+-------------+---------------------+----------+
|customer_id|customer_name|food_delivery_address|sales_date|
+-----------+-------------+---------------------+----------+
|1          |Ram          |Bangalore            |2026-04-08|
|2          |Shyam        |Mumbai               |2026-04-08|
|3          |Mohan        |Delhi                |2026-04-08|
|4          |Sita         |Pune                 |2026-04-08|
|5          |Geeta        |Hyderabad            |2026-04-08|
|6          |Ravi         |Delhi                |2026-04-08|
|7          |Amit         |Noida                |2026-04-08|
|8          |Neha         |Gurgaon              |2026-04-08|
+-----------+-------------+---------------------+----------+



In [ ]:
joined_data = source_df.alias("src").join(target_df.alias("tgt"),col("src.customer_id") == col("tgt.customer_id"),"left")
joined_data.show()

+-----------+-------------+---------------------+----------+-----------+-------------+---------+--------------------+------------------+------+
|customer_id|customer_name|food_delivery_address|sales_date|customer_id|customer_name|     city|effective_start_date|effective_end_date|active|
+-----------+-------------+---------------------+----------+-----------+-------------+---------+--------------------+------------------+------+
|          1|          Ram|            Bangalore|2026-04-08|          1|          Ram|   Mumbai|          2024-01-01|              NULL|     Y|
|          1|          Ram|            Bangalore|2026-04-08|          1|          Ram|    Delhi|          2023-01-01|        2024-01-01|     N|
|          3|        Mohan|                Delhi|2026-04-08|          3|        Mohan|    Delhi|          2024-06-01|              NULL|     Y|
|          3|        Mohan|                Delhi|2026-04-08|          3|        Mohan|     Pune|          2023-06-01|        2024-06-01|

In [ ]:
#New customers
new_customers = joined_data.filter(col("tgt.customer_id").isNull())
new_customers.show()

+-----------+-------------+---------------------+----------+-----------+-------------+----+--------------------+------------------+------+
|customer_id|customer_name|food_delivery_address|sales_date|customer_id|customer_name|city|effective_start_date|effective_end_date|active|
+-----------+-------------+---------------------+----------+-----------+-------------+----+--------------------+------------------+------+
|          8|         Neha|              Gurgaon|2026-04-08|       NULL|         NULL|NULL|                NULL|              NULL|  NULL|
|          7|         Amit|                Noida|2026-04-08|       NULL|         NULL|NULL|                NULL|              NULL|  NULL|
+-----------+-------------+---------------------+----------+-----------+-------------+----+--------------------+------------------+------+



In [ ]:
# Changed Records
changed_records = joined_data.filter(
    (col("tgt.customer_id").isNotNull()) &
    (col("tgt.active") == "Y") &
        (
        col("src.food_delivery_address").isNull() != col("tgt.city").isNull() |
        (col("src.food_delivery_address").isNotNull() & col("tgt.city").isNotNull() &
         (col("src.food_delivery_address") != col("tgt.city")))
        )
)
changed_records.show()

+-----------+-------------+---------------------+----------+-----------+-------------+---------+--------------------+------------------+------+
|customer_id|customer_name|food_delivery_address|sales_date|customer_id|customer_name|     city|effective_start_date|effective_end_date|active|
+-----------+-------------+---------------------+----------+-----------+-------------+---------+--------------------+------------------+------+
|          1|          Ram|            Bangalore|2026-04-08|          1|          Ram|   Mumbai|          2024-01-01|              NULL|     Y|
|          4|         Sita|                 Pune|2026-04-08|          4|         Sita|  Chennai|          2024-03-01|              NULL|     Y|
|          6|         Ravi|                Delhi|2026-04-08|          6|         Ravi|Bangalore|          2024-07-01|              NULL|     Y|
+-----------+-------------+---------------------+----------+-----------+-------------+---------+--------------------+------------------+

In [ ]:
unchanged_records = joined_data.filter(
    (col("tgt.customer_id").isNotNull()) &
    (col("tgt.active") == "Y") &
    (
        (col("src.food_delivery_address").isNull() & col("tgt.city").isNull()) |
        (col("src.food_delivery_address").isNotNull() & col("tgt.city").isNotNull() &
         (col("src.food_delivery_address") == col("tgt.city")))
    )
)
unchanged_records.show()

+-----------+-------------+---------------------+----------+-----------+-------------+---------+--------------------+------------------+------+
|customer_id|customer_name|food_delivery_address|sales_date|customer_id|customer_name|     city|effective_start_date|effective_end_date|active|
+-----------+-------------+---------------------+----------+-----------+-------------+---------+--------------------+------------------+------+
|          2|        Shyam|               Mumbai|2026-04-08|          2|        Shyam|   Mumbai|          2024-01-01|              NULL|     Y|
|          3|        Mohan|                Delhi|2026-04-08|          3|        Mohan|    Delhi|          2024-06-01|              NULL|     Y|
|          5|        Geeta|            Hyderabad|2026-04-08|          5|        Geeta|Hyderabad|          2024-02-01|              NULL|     Y|
+-----------+-------------+---------------------+----------+-----------+-------------+---------+--------------------+------------------+

In [ ]:
new_customer_df = new_customers.select(
    col("src.customer_id"),
    col("src.customer_name"),
    col("src.food_delivery_address").alias("city"),
    lit("Y").alias("active"),
    col("src.sales_date").alias("effective_start_date"),
    lit(None).alias("effective_end_date")
)
new_customer_df.show()

+-----------+-------------+-------+------+--------------------+------------------+
|customer_id|customer_name|   city|active|effective_start_date|effective_end_date|
+-----------+-------------+-------+------+--------------------+------------------+
|          8|         Neha|Gurgaon|     Y|          2026-04-08|              NULL|
|          7|         Amit|  Noida|     Y|          2026-04-08|              NULL|
+-----------+-------------+-------+------+--------------------+------------------+



In [ ]:
expired_old_df = changed_records.select(
    col("tgt.customer_id"),
    col("tgt.customer_name"),
    col("tgt.city"),
    lit("N").alias("active"),
    col("tgt.effective_start_date"),
    col("src.sales_date").alias("effective_end_date")
)
expired_old_df.show()

+-----------+-------------+---------+------+--------------------+------------------+
|customer_id|customer_name|     city|active|effective_start_date|effective_end_date|
+-----------+-------------+---------+------+--------------------+------------------+
|          1|          Ram|   Mumbai|     N|          2024-01-01|        2026-04-08|
|          4|         Sita|  Chennai|     N|          2024-03-01|        2026-04-08|
|          6|         Ravi|Bangalore|     N|          2024-07-01|        2026-04-08|
+-----------+-------------+---------+------+--------------------+------------------+



In [ ]:
new_version_df = changed_records.select(
    col("src.customer_id"),
    col("src.customer_name"),
    col("src.food_delivery_address").alias("city"),
    lit("Y").alias("active"),
    col("src.sales_date").alias("effective_start_date"),
    lit(None).alias("effective_end_date")
)
new_version_df.show()

+-----------+-------------+---------+------+--------------------+------------------+
|customer_id|customer_name|     city|active|effective_start_date|effective_end_date|
+-----------+-------------+---------+------+--------------------+------------------+
|          1|          Ram|Bangalore|     Y|          2026-04-08|              NULL|
|          4|         Sita|     Pune|     Y|          2026-04-08|              NULL|
|          6|         Ravi|    Delhi|     Y|          2026-04-08|              NULL|
+-----------+-------------+---------+------+--------------------+------------------+



In [ ]:
#Keep Unchanged Records
unchanged_df = unchanged_records.select(
    col("tgt.customer_id"),
    col("tgt.customer_name"),
    col("tgt.city"),
    col("tgt.active"),
    col("tgt.effective_start_date"),
    col("tgt.effective_end_date")
)
unchanged_df.show()

+-----------+-------------+---------+------+--------------------+------------------+
|customer_id|customer_name|     city|active|effective_start_date|effective_end_date|
+-----------+-------------+---------+------+--------------------+------------------+
|          2|        Shyam|   Mumbai|     Y|          2024-01-01|              NULL|
|          3|        Mohan|    Delhi|     Y|          2024-06-01|              NULL|
|          5|        Geeta|Hyderabad|     Y|          2024-02-01|              NULL|
+-----------+-------------+---------+------+--------------------+------------------+



In [ ]:
final_df = unchanged_df \
    .unionByName(expired_old_df) \
    .unionByName(new_version_df) \
    .unionByName(new_customer_df)

final_df.orderBy(col('customer_id').asc(),col('effective_start_date').asc()).show(truncate=False)

+-----------+-------------+---------+------+--------------------+------------------+
|customer_id|customer_name|city     |active|effective_start_date|effective_end_date|
+-----------+-------------+---------+------+--------------------+------------------+
|1          |Ram          |Mumbai   |N     |2024-01-01          |2026-04-08        |
|1          |Ram          |Bangalore|Y     |2026-04-08          |NULL              |
|2          |Shyam        |Mumbai   |Y     |2024-01-01          |NULL              |
|3          |Mohan        |Delhi    |Y     |2024-06-01          |NULL              |
|4          |Sita         |Chennai  |N     |2024-03-01          |2026-04-08        |
|4          |Sita         |Pune     |Y     |2026-04-08          |NULL              |
|5          |Geeta        |Hyderabad|Y     |2024-02-01          |NULL              |
|6          |Ravi         |Bangalore|N     |2024-07-01          |2026-04-08        |
|6          |Ravi         |Delhi    |Y     |2026-04-08          |

In [ ]:
final_df.createOrReplaceTempView("Customer")

In [ ]:
df_sql = spark.sql("SELECT * FROM Customer order by 1,5")
df_sql.show()

+-----------+-------------+---------+------+--------------------+------------------+
|customer_id|customer_name|     city|active|effective_start_date|effective_end_date|
+-----------+-------------+---------+------+--------------------+------------------+
|          1|          Ram|   Mumbai|     N|          2024-01-01|        2026-04-08|
|          1|          Ram|Bangalore|     Y|          2026-04-08|              NULL|
|          2|        Shyam|   Mumbai|     Y|          2024-01-01|              NULL|
|          3|        Mohan|    Delhi|     Y|          2024-06-01|              NULL|
|          4|         Sita|  Chennai|     N|          2024-03-01|        2026-04-08|
|          4|         Sita|     Pune|     Y|          2026-04-08|              NULL|
|          5|        Geeta|Hyderabad|     Y|          2024-02-01|              NULL|
|          6|         Ravi|Bangalore|     N|          2024-07-01|        2026-04-08|
|          6|         Ravi|    Delhi|     Y|          2026-04-08|